# Dental Vision V1 — V3 Failure Audit
No retraining. Audits V3 held-out predictions to identify the failure mechanism before V4: score distribution, IoU/GT matching, duplicate predictions, class confusion, box geometry, and out-of-image annotations.


In [ ]:
import os,shutil,pathlib,json,zipfile,glob,random,math
%cd /kaggle/working
shutil.rmtree('/kaggle/working/dental-vision-v1',ignore_errors=True)
!git clone https://github.com/drhaidarali95/dental-vision-v1.git /kaggle/working/dental-vision-v1
%cd /kaggle/working/dental-vision-v1
!pip -q install -r requirements.txt matplotlib


In [ ]:
cands=glob.glob('/kaggle/input/**/dentex_holdout_v3.pt',recursive=True)
print('CHECKPOINT CANDIDATES:',cands)
assert cands, 'Attach the successful V3 output containing dentex_holdout_v3.pt as Kaggle Input.'
ckpt=cands[0]; print('USING:',ckpt)


In [ ]:
root=pathlib.Path('data/dentex_failure_audit'); shutil.rmtree(root,ignore_errors=True); root.mkdir(parents=True)
!python scripts/download_dentex.py --out data/dentex_failure_audit --files training_data.zip
zpath=root/'training_data.zip'; wanted={'caries','deep caries','periapical lesion','periapical lesions','impacted','impacted tooth','impacted teeth'}; candidates=[]
with zipfile.ZipFile(zpath) as z:
    for name in z.namelist():
        if not name.lower().endswith('.json'): continue
        try: d0=json.loads(z.read(name))
        except Exception: continue
        if not isinstance(d0,dict) or not {'images','annotations'}.issubset(d0): continue
        cats=d0.get('categories_3') or d0.get('categories') or []; names={str(c.get('name','')).strip().lower() for c in cats}; score=len(names&wanted); bonus=2 if 'quadrant-enumeration-disease' in name.lower() else 0
        if score or bonus: candidates.append((score+bonus,len(d0['images']),name,d0,cats))
assert candidates
_,_,ann_member,d,cats=max(candidates,key=lambda x:(x[0],x[1])); d['categories']=cats
for a in d['annotations']:
    if 'category_id_3' in a: a['category_id']=a['category_id_3']
subset=root/'diagnostic'; (subset/'images').mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(zpath) as z:
    members=z.namelist()
    for im in d['images']:
        fn=str(im['file_name']).replace('\\','/').lstrip('./'); matches=[m for m in members if m.endswith('/'+fn) or m==fn] or [m for m in members if pathlib.PurePosixPath(m).name==pathlib.PurePosixPath(fn).name]; src=matches[0]; dest=subset/'images'/pathlib.PurePosixPath(fn).name
        with z.open(src) as r,open(dest,'wb') as w: shutil.copyfileobj(r,w)
        im['file_name']=dest.name
(subset/'all.json').write_text(json.dumps(d)); zpath.unlink()
!python scripts/split_dentex_holdout.py --annotations data/dentex_failure_audit/diagnostic/all.json --train-out data/dentex_failure_audit/diagnostic/train.json --val-out data/dentex_failure_audit/diagnostic/val.json --val-fraction 0.20 --seed 20260920


In [ ]:
import torch, numpy as np
from torchvision.transforms import v2 as T
from torchvision.ops import box_iou
from train import build_model
from src.dentex_dataset import DentexCocoDataset
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ds=DentexCocoDataset('data/dentex_failure_audit/diagnostic/images','data/dentex_failure_audit/diagnostic/val.json',transforms=T.Compose([T.ToImage(),T.ToDtype(torch.float32,scale=True)]))
state=torch.load(ckpt,map_location=device,weights_only=False); model=build_model(state['num_classes']).to(device); model.load_state_dict(state['model']); model.eval()
label_names={k:v.get('name',str(k)) for k,v in ds.label_to_category.items()}; print('LABELS:',label_names)


In [ ]:
# Full held-out audit at several score thresholds.
thresholds=[0.01,0.05,0.10,0.25,0.50]
stats={str(th):{'pred':0,'matched':0,'dup':0,'images':0} for th in thresholds}; score_pool=[]; per_image=[]; gt_oob=[]; confusion={}
for i in range(len(ds)):
    x,t=ds[i]; info=ds.images[ds.ids[i]]; W,H=int(info.get('width',x.shape[-1])),int(info.get('height',x.shape[-2]))
    gt=t['boxes'].cpu(); gl=t['labels'].cpu()
    for j,b in enumerate(gt):
        x1,y1,x2,y2=[float(v) for v in b]
        if x1<0 or y1<0 or x2>W or y2>H or x2<=x1 or y2<=y1: gt_oob.append({'file':info['file_name'],'gt_index':j,'box':[x1,y1,x2,y2],'image_wh':[W,H]})
    with torch.no_grad(): p=model([x.to(device)])[0]
    pb=p['boxes'].detach().cpu(); pl=p['labels'].detach().cpu(); ps=p['scores'].detach().cpu(); score_pool.extend([float(s) for s in ps])
    rec={'file':info['file_name'],'gt':len(gt),'thresholds':{}}
    for th in thresholds:
        keep=ps>=th; b=pb[keep]; l=pl[keep]; s=ps[keep]; n=len(b); matched=0; dup=0; wrong_class=0
        if len(gt) and n:
            I=box_iou(b,gt)
            best_iou,best_gt=I.max(dim=1)
            good=best_iou>=0.5; matched=int(good.sum())
            for k in torch.where(good)[0].tolist():
                g=int(best_gt[k]); key=f'{int(gl[g])}->{int(l[k])}'; confusion[key]=confusion.get(key,0)+1
                if int(gl[g])!=int(l[k]): wrong_class+=1
            # Duplicate = extra same-class predictions overlapping the same GT at IoU>=0.5.
            for g in range(len(gt)):
                hits=((I[:,g]>=0.5)&(l==gl[g])).sum().item(); dup+=max(0,int(hits)-1)
        stats[str(th)]['pred']+=n; stats[str(th)]['matched']+=matched; stats[str(th)]['dup']+=dup; stats[str(th)]['images']+=1
        rec['thresholds'][str(th)]={'pred':n,'iou50_predictions':matched,'duplicate_same_class_iou50':dup,'wrong_class_among_iou50':wrong_class}
    per_image.append(rec)
score_pool=np.array(score_pool,dtype=float)
score_quantiles={str(q):float(np.quantile(score_pool,q)) for q in [0,.1,.25,.5,.75,.9,.95,.99,1]} if len(score_pool) else {}
report={'images':len(ds),'gt_boxes':sum(x['gt'] for x in per_image),'score_quantiles':score_quantiles,'threshold_summary':stats,'gt_out_of_bounds':gt_oob,'iou50_class_pairs':confusion,'per_image':per_image}
path='/kaggle/working/v3_failure_audit.json'; json.dump(report,open(path,'w'),indent=2)
print('SCORE QUANTILES:',score_quantiles)
print('THRESHOLD SUMMARY:',json.dumps(stats,indent=2))
print('OUT-OF-BOUNDS GT:',len(gt_oob))
print('IOU>=0.5 CLASS PAIRS:',confusion)
print('WROTE',path)


In [ ]:
# Compact diagnosis table: precision-like localization signal and prediction burden.
rows=[]
for th in thresholds:
    s=stats[str(th)]; rows.append({'threshold':th,'predictions':s['pred'],'iou50_predictions':s['matched'],'iou50_fraction_of_predictions':round(s['matched']/max(s['pred'],1),4),'duplicates':s['dup'],'avg_predictions_per_image':round(s['pred']/len(ds),2)})
import pandas as pd
display(pd.DataFrame(rows))
print('GT boxes:',report['gt_boxes'],'| OOB GT:',len(gt_oob))
print('Send v3_failure_audit.json and this table to ChatGPT. Do NOT retrain V4 yet.')
